In [14]:
import pandas as pd
import numpy as np
import sklearn
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


In [15]:
# In case of errors, check if you are using the correct versions of the libraries. This notebook works on the following versions of libraries. Specify these versions if you get version related errors.
# TensorFlow version: 2.17.0
# Pandas version: 2.1.4
# NumPy version: 1.26.4
# Scikit-learn version: 1.5.2
# Keras version: 3.4.1

# print("TensorFlow version:", tf.__version__)
# print("Pandas version:", pd.__version__)
# print("NumPy version:", np.__version__)
# print("Scikit-learn version:", sklearn.__version__)
# print("Keras version:", tf.keras.__version__)

## Part 1: Import the Housing data and do feature transformations

In [16]:
df= pd.read_csv('house-price-full.csv')
df.head()

,bedrooms,sqft_living,price
0,3,1340,313000
1,5,3650,2384000
2,3,1930,342000
3,3,2000,420000
4,4,1940,550000


In [17]:
X = df.copy()
# Remove target
Y = X.pop('price')

# perform a scaler transform of the input data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# perform log transformation of target variable (For Sandeep: Is this needed?)
Y = np.log(Y)

In [18]:
df_scaled = pd.DataFrame(X)
df_scaled

,0,1
0,-0.433198,-0.753258
1,1.675735,1.457330
2,-0.433198,-0.188649
3,-0.433198,-0.121661
4,0.621269,-0.179079
...,...,...
494,0.621269,0.873582
495,1.675735,2.299459
496,-0.433198,-0.724549
497,-0.433198,-0.179079


In [20]:
X_train,X_val,y_train,y_val=train_test_split(df_scaled,Y,test_size=0.1,random_state=42)

In [21]:
Y

0      12.653958
1      14.684290
2      12.742566
3      12.948010
4      13.217674
         ...    
494    13.380102
495    13.764217
496    12.128111
497    12.721886
498    12.254863
Name: price, Length: 499, dtype: float64

## Part 2: Create Model Using `keras`

![](multiple_neurons.png)

In [7]:
from tensorflow import keras

In [22]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=(X.shape[-1],)),
        keras.layers.Dense(
            10, activation="relu"
        ),

        keras.layers.Dense(
            10, activation="relu"
        ),
        keras.layers.Dense(
            5, activation="relu"
        ),
        keras.layers.Dense(1, activation="linear")
    ]
)
 

In [23]:
model.summary()
model.compile(
    optimizer='sgd',          # or any other optimizer like 'sgd', 'rmsprop', etc.
    loss='mse',                # Mean Squared Error as loss function
    metrics=['mse']            # Mean Squared Error as evaluation metric
)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                     │ (None, 10)                  │              30 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 10)                  │             110 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_12 (Dense)                     │ (None, 5)                   │              55 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_13 (Dense)                     │ (None, 1)                   │               6 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 201 (804.00 B)

 Trainable params: 201 (804.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.fit(X_train,y_train,epochs=10,batch_size=32,validation_data=(X_val,y_val),verbose=1)

Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - loss: 48.0176 - mse: 48.0176 - val_loss: 1.1076 - val_mse: 1.1076
Epoch 2/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 4.7518 - mse: 4.7518 - val_loss: 4.3134 - val_mse: 4.3134
Epoch 3/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 3.9798 - mse: 3.9798 - val_loss: 4.2652 - val_mse: 4.2652
Epoch 4/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 2.2333 - mse: 2.2333 - val_loss: 0.4386 - val_mse: 0.4386
Epoch 5/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.7246 - mse: 0.7246 - val_loss: 0.5695 - val_mse: 0.5695
Epoch 6/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.3864 - mse: 0.3864 - val_loss: 3.9148 - val_mse: 3.9148
Epoch 7/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 1.0282 - mse: 1.0282 - val_loss: 0.1461 - val_mse: 0.1461
Epoch 8/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.2046 - mse: 0.2046 - val_loss: 2.2511 - val_mse: 2.2511
Epoch 9/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.7

In [ ]:
# Epoch 10/10
# 15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.2016 - mse: 0.2016 - val_loss: 0.1436 - val_mse: 0.1436

In [25]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=(X.shape[-1],)),
        keras.layers.Dense(
            10, activation="relu"
        ),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(
            10, activation="relu"
        ),
        keras.layers.Dense(
            5, activation="relu"
        ),
        keras.layers.Dense(1, activation="linear")
    ]
)
 

In [26]:
model.summary()
model.compile(
    optimizer='sgd',          # or any other optimizer like 'sgd', 'rmsprop', etc.
    loss='mse',                # Mean Squared Error as loss function
    metrics=['mse']            # Mean Squared Error as evaluation metric
)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                     │ (None, 10)                  │              30 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 10)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_15 (Dense)                     │ (None, 10)                  │             110 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_16 (Dense)                     │ (None, 5)                   │              55 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (None, 1)                   │               6 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 201 (804.00 B)

 Trainable params: 201 (804.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.fit(X_train,y_train,epochs=10,batch_size=32,validation_data=(X_val,y_val),verbose=1)

Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - loss: 38.6042 - mse: 38.6042 - val_loss: 17.8996 - val_mse: 17.8996
Epoch 2/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 4.6177 - mse: 4.6177 - val_loss: 5.2154 - val_mse: 5.2154
Epoch 3/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 2.1286 - mse: 2.1286 - val_loss: 2.2732 - val_mse: 2.2732
Epoch 4/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.8313 - mse: 0.8313 - val_loss: 0.6815 - val_mse: 0.6815
Epoch 5/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.4808 - mse: 0.4808 - val_loss: 0.3620 - val_mse: 0.3620
Epoch 6/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.2491 - mse: 0.2491 - val_loss: 0.3504 - val_mse: 0.3504
Epoch 7/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.3945 - mse: 0.3945 - val_loss: 0.6232 - val_mse: 0.6232
Epoch 8/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.3304 - mse: 0.3304 - val_loss: 3.2957 - val_mse: 3.2957
Epoch 9/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0

In [ ]:
# Epoch 10/10
# 15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.2016 - mse: 0.2016 - val_loss: 0.1436 - val_mse: 0.1436


# after drop out decrease in val loss and val mse
# Epoch 10/10
# 15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2680 - mse: 0.2680 - val_loss: 0.5719 - val_mse: 0.5719
